# Theme: Transformer - Multi-Head Attention 구현 & BERT 감성 분석 실습

이 노트북에서는 트랜스포머(Transformer) 모델의 병렬 정보 처리 핵심 메커니즘인 **Multi-Head Attention**을 PyTorch로 직접 구현하여 동작 원리를 시각화하고, 이를 바탕으로 Hugging Face의 `transformers` 라이브러리를 활용해 한국어 사전학습 모델(`klue/bert-base`)을 로드하여 감성 분석(Sentiment Analysis)을 수행하는 실무 파이프라인을 구축합니다.

현업 수준의 신뢰성 높은 머신러닝 파이프라인 설계를 위해 입력 데이터의 무결성을 검증하는 **Pandera** 라이브러리도 함께 연계하여 실습합니다.

### 학습 목표
1. **Multi-Head Attention**의 선형 투영(Linear Projection), 텐서 분할(Tensor Split), Scaled Dot-Product 연산 및 병합(Concatenation) 프로세스를 코드로 직접 구현하여 동작 차원을 완벽하게 이해합니다.
2. **Pandera**를 사용해 NLP 파이프라인 입력단에서 데이터 스키마와 데이터 유효성을 엄격하게 제어하는 실무형 코드를 학습합니다.
3. **Hugging Face Transformers**의 `AutoTokenizer` 및 `AutoModelForSequenceClassification`을 활용하여 전이학습(Fine-tuning)을 위한 기본 데이터 가공 및 학습 루프를 PyTorch로 직접 작성합니다.

In [10]:
# %pip -q install scikit-learn tqdm emoji pandera
# %pip -q install -U "transformers>=4.44,<4.47" "datasets>=2.20" "accelerate>=0.34" "evaluate>=0.4"

import torch
import evaluate
import joblib
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import numpy as np
import pandas as pd
import os
from datasets import Dataset, DatasetDict
import re
import pandera.pandas as pa
from pandera.typing import Series
from torch.utils.data import Dataset as TDataset, DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline, BartForConditionalGeneration, PreTrainedTokenizerFast
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import sentencepiece as spm

# 재현성을 위한 시드 고정
torch.manual_seed(42)
np.random.seed(42)

---
## 1. Multi-Head Attention 직접 구현

Multi-Head Attention은 입력 시퀀스에 대해 단일 어텐션을 적용하는 대신, Query, Key, Value를 서로 다른 고유한 $h$개의 헤드(Heads)로 프로젝션(Projection)하여 어텐션 연산을 병렬로 처리합니다. 이를 통해 모델은 서로 다른 문맥 및 의미적 측면에서 관계를 파악할 수 있는 유연성을 확보합니다.

### 핵심 수식
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$$
$$\text{where } \text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

여기서 각각의 사영 행렬의 차원은 다음과 같습니다:
* $W_i^Q \in \mathbb{R}^{d_{model} \times d_k}$
* $W_i^K \in \mathbb{R}^{d_{model} \times d_k}$
* $W_i^V \in \mathbb{R}^{d_{model} \times d_v}$
* $W^O \in \mathbb{R}^{h d_v \times d_{model}}$

보통 연산의 편의성과 파라미터 효율성을 위해 각 헤드의 타겟 차원 $d_k$와 $d_v$는 $d_{model} / h$로 설정합니다.

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """6월 4일 실습에서 구현한 기본 Scaled Dot-Product Attention 클래스"""
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()
        
    def forward(self, Q, K, V, mask=None):
        # Q, K, V 차원: [Batch_size, num_heads, Seq_len, d_k]
        d_k = Q.size(-1)
        
        # 1. Q @ K^T (마지막 두 개의 차원을 전치하여 내적 연산 수행)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        
        # 2. 마스킹 적용
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        # 3. Softmax를 통한 어텐션 가중치 확률 맵 생성
        attn_weights = F.softmax(scores, dim=-1)
        
        # 4. 어텐션 가중치에 Value 행렬을 가중합 연산 수행
        context = torch.matmul(attn_weights, V)
        
        return context, attn_weights

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        
        # d_model은 헤드의 수(num_heads)로 나누어 떨어져야 함
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_k = d_model // num_heads
        
        # Q, K, V의 Linear 사영 레이어 정의
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # Scaled Dot-Product 어텐션 모듈
        self.attention = ScaledDotProductAttention()
        
        # 최종 아웃풋 사영 레이어 (W^O)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        
        # 1. Linear Projection 진행
        # Output size: [Batch_size, Seq_len, d_model]
        q_proj = self.W_q(Q)
        k_proj = self.W_k(K)
        v_proj = self.W_v(V)
        
        # 2. 텐서 형태 변환 및 분할 (Split into multiple heads)
        # [Batch_size, Seq_len, d_model] -> [Batch_size, Seq_len, num_heads, d_k] -> [Batch_size, num_heads, Seq_len, d_k]
        q_heads = q_proj.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k_heads = k_proj.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v_heads = v_proj.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. 마스크 차원 확장 (배치 크기와 헤드 수에 맞춤)
        if mask is not None:
            # mask shape: [Batch_size, 1, 1, Seq_len] or [Batch_size, 1, Seq_len, Seq_len]
            mask = mask.unsqueeze(1)
            
        # 4. Scaled Dot-Product Attention 병렬 계산
        # context shape: [Batch_size, num_heads, Seq_len, d_k]
        # attn_weights shape: [Batch_size, num_heads, Seq_len, Seq_len]
        context, attn_weights = self.attention(q_heads, k_heads, v_heads, mask)
        
        # 5. 헤드 병합 (Concatenate Heads)
        # [Batch_size, num_heads, Seq_len, d_k] -> [Batch_size, Seq_len, num_heads, d_k]
        # -> [Batch_size, Seq_len, d_model (num_heads * d_k)]
        context_concat = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 6. 최종 Output Linear 사영 적용
        output = self.W_o(context_concat)
        
        return output, attn_weights

### 모의 데이터로 Multi-Head Attention 차원 변환 확인하기
가상의 문장 데이터와 임베딩을 구성하여 구현한 Multi-Head Attention이 정상적으로 차원을 연산하고 처리하는지 검증합니다.
* 가상 문장 길이: 6 단어
* 임베딩 차원 ($d_{model}$): 64
* 헤드 개수 ($num\_heads$): 8개

In [ ]:
# 하이퍼파라미터 정의
batch_size = 2
seq_len = 6
d_model = 64
num_heads = 8

# 가상의 입력 텐서 생성
x = torch.randn(batch_size, seq_len, d_model)

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
output, weights = mha(x, x, x) # Self-Attention 연산

print("===== Multi-Head Attention 검증 =====")
print(f"입력 텐서 크기: {x.shape}")
print(f"출력 텐서 크기: {output.shape} (입력과 동일해야 함)")
print(f"어텐션 가중치 크기: {weights.shape} -> [Batch, Heads, Seq_len, Seq_len]")

===== Multi-Head Attention 검증 =====
입력 텐서 크기: torch.Size([2, 6, 64])
출력 텐서 크기: torch.Size([2, 6, 64]) (입력과 동일해야 함)
어텐션 가중치 크기: torch.Size([2, 8, 6, 6]) -> [Batch, Heads, Seq_len, Seq_len]


----- 
## 2. Pandera를 활용한 데이터 무결성 검증

실전 데이터 분석 및 ML 엔지니어링 파이프라인에서 가장 잦은 에러는 입력 데이터의 누락, 타입 오류, 혹은 범위를 벗어난 이상치 라벨 등 데이터 품질 관련 이슈입니다.
`pandera`는 Pandas 데이터프레임의 스키마를 런타임에 완격하게 검증하여 파이프라인을 견고하게 만듭니다.

In [ ]:
# Pandera 데이터 스키마 정의
class SentimentSchema(pa.DataFrameModel):
    # 리뷰 텍스트 컬럼: 문자열 형식이어야 하며, 결측치가 없어야 함
    review_text: Series[str] = pa.Field(coerce=True, nullable=False)
    # 감성 분석 라벨: 정수 형식이며, [0, 1] 범위여야 함 (0: 부정, 1: 긍정)
    label: Series[int] = pa.Field(coerce=True, check_name=True, isin=[0, 1])

    class Config:
        strict = True  # 스키마에 정의되지 않은 추가 컬럼 검출 차단
        coerce = True  # 자동 타입 변환 활성화

# 가상의 실습용 데이터 프레임 생성
mock_raw_data = {
    "review_text": [
        "음식이 너무 맛있고 배달도 진짜 빨라요! 대만족",
        "양이 너무 적어서 실망했어요. 다음엔 안 시킬 듯",
        "리뷰 보고 주문했는데 보통이네요.",
        "진짜 최악입니다. 머리카락 나옴",
        "리뷰 이벤트 참여요! 서비스 감사합니다."
    ],
    "label": [1, 0, 0, 0, 1] # 1: 긍정, 0: 부정
}

df_clean = pd.DataFrame(mock_raw_data)

try:
    # 데이터 검증 시도
    validated_df = SentimentSchema.validate(df_clean)
    print("✅ 데이터 스키마 검증을 통과했습니다!")
    print(validated_df.head())
except pa.errors.SchemaError as e:
    print(f"❌ 데이터 스키마 검증 실패:\n{e}")

✅ 데이터 스키마 검증을 통과했습니다!
                   review_text  label
0   음식이 너무 맛있고 배달도 진짜 빨라요! 대만족      1
1  양이 너무 적어서 실망했어요. 다음엔 안 시킬 듯      0
2           리뷰 보고 주문했는데 보통이네요.      0
3            진짜 최악입니다. 머리카락 나옴      0
4       리뷰 이벤트 참여요! 서비스 감사합니다.      1


### 데이터 검증 실패 케이스 강제 확인하기
비정상 라벨(`2`)이나 결측치(`None`)가 데이터에 혼입되었을 때, `pandera`가 어떻게 런타임에 에러를 감지하고 시스템 다운스트림으로의 오류 전파를 차단하는지 입증합니다.

In [ ]:
# 유효하지 않은 라벨과 결측치가 포함된 불량 데이터프레임 생성
bad_raw_data = {
    "review_text": ["맛있어요!", None, "그냥 그래요"],
    "label": [1, 0, 2] # 2는 정의되지 않은 범위를 벗어난 이상치 라벨
}
df_bad = pd.DataFrame(bad_raw_data)

try:
    SentimentSchema.validate(df_bad)
except pa.errors.SchemaError as e:
    print("❌ Pandera 에러 자동 탐지 성공:")
    print(e)

❌ Pandera 에러 자동 탐지 성공:
non-nullable series 'review_text' contains null values:
1    NaN
Name: review_text, dtype: str


---
## 3. Hugging Face Transformers 활용 BERT 감성 분석

이제 데이터 스키마 검증을 마친 데이터프레임을 토대로 사전 학습된 한국어 BERT 모델을 이용하여 학습용 Dataset 클래스를 구축하고, 훈련을 진행할 수 있는 데이터 파이프라인을 구축합니다.

In [ ]:
# 1. 토크나이저 및 사전학습 모델 로드
# 한국어 대표적 벤치마크인 klue의 klue/bert-base 모델을 활용합니다.
MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"모델이 로드된 디바이스: {device}")

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델이 로드된 디바이스: cpu


In [ ]:
# 2. PyTorch 전용 커스텀 데이터셋 구현
class ReviewDataset(TDataset):
    def __init__(self, dataframe, tokenizer, max_len=64):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        review = str(row["review_text"])
        label = int(row["label"])
        
        # BERT 입력을 위한 인코딩 수행
        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].squeeze(0),         # [max_len]
            "attention_mask": encoding["attention_mask"].squeeze(0), # [max_len]
            "label": torch.tensor(label, dtype=torch.long)          # [1]
        }

# 데이터 로더 구축
train_dataset = ReviewDataset(validated_df, tokenizer, max_len=32)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# 하나의 미니배치 샘플 규격 출력
sample_batch = next(iter(train_loader))
print("===== 데이터로더 미니배치 형태 확인 =====")
print("input_ids shape:", sample_batch["input_ids"].shape)
print("attention_mask shape:", sample_batch["attention_mask"].shape)
print("label shape:", sample_batch["label"].shape)

===== 데이터로더 미니배치 형태 확인 =====
input_ids shape: torch.Size([2, 32])
attention_mask shape: torch.Size([2, 32])
label shape: torch.Size([2])


### BERT 모델의 원-스텝 학습 및 손실함수 계산 실습
실제로 모델이 학습 루프 내에서 가중치 그래디언트를 역전파(Backpropagation)하기 위해 거치는 forward 연산 및 loss 계산 흐름을 추적합니다.

In [ ]:
# 옵티마이저 정의
optimizer = AdamW(model.parameters(), lr=2e-5)

# 간단한 원-스텝 학습 루프 시뮬레이션
model.train()

print("===== BERT 훈련(Forward & Backpropagation) 루프 테스트 =====")
for step, batch in enumerate(train_loader):
    # 데이터를 디바이스로 이동
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)
    
    # 기울기 초기화
    optimizer.zero_grad()
    
    # Forward 연산
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    
    # Hugging Face 모델은 labels를 넘겨주면 내부적으로 loss를 계산하여 반환합니다.
    loss = outputs.loss
    logits = outputs.logits
    
    # 역전파 및 가중치 업데이트
    loss.backward()
    optimizer.step()
    
    print(f"Step {step + 1} | Loss: {loss.item():.4f} | Logits: {logits.detach().cpu().numpy()}")

===== BERT 훈련(Forward & Backpropagation) 루프 테스트 =====
Step 1 | Loss: 0.3032 | Logits: [[-0.23690939  0.7153909 ]
 [-0.44188982  0.68708515]]
Step 2 | Loss: 1.1076 | Logits: [[-0.04045611  0.5572273 ]
 [-0.11486237  0.6969341 ]]
Step 3 | Loss: 1.4275 | Logits: [[-0.73627704  0.41692847]]


---
## 4. 실습 과제
1. 본 실습에서 구현한 `MultiHeadAttention` 모델에 대해 임의의 3차원 텐서를 가상의 입력값으로 전달하고, Multi-Head 어텐션 동작 과정에서 `attn_weights`에 저장되는 각 헤드별 어텐션 맵을 추출한 다음 matplotlib 히트맵으로 시각화해 보세요.
2. 만약 BERT의 토크나이저 아웃풋 결과물인 패딩된 문장들에 대해 마스킹이 적용되어야 할 때, `MultiHeadAttention` 내부의 `mask` 연산이 어떻게 작동할 수 있는지 가상의 마스크(`[Batch, 1, 1, Seq_len]`)를 정의하고 연산 결과를 확인하세요.

In [ ]:
train_path = '../content/ratings_train.txt'
test_path = '../content/ratings_test.txt'

# import os
# # 데이터 다운로드
# !rm -rf nsmc && git clone -q https://github.com/e9t/nsmc.git
# data_dir = "/content/nsmc"
# train_path = os.path.join(data_dir, '/content/nsmc/ratings_train.txt')
# test_path  = os.path.join(data_dir, '/content/nsmc/ratings_test.txt')

train_df = pd.read_csv(train_path, sep='\t')
test_df  = pd.read_csv(test_path,  sep='\t')

def clean_korean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # 한글, 숫자, 기본 문장부호만 남기기
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.\,\!\?\-~…]", " ", text)
    # 연속 공백 정리
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 결측/공백 제거
train_df = train_df.dropna(subset=['document']).copy()
test_df  = test_df.dropna(subset=['document']).copy()
train_df['document'] = train_df['document'].map(clean_korean)
test_df['document']  = test_df['document'].map(clean_korean)

# 빈 문자열 제거
train_df = train_df[train_df['document'].str.len() > 0]
test_df  = test_df[test_df['document'].str.len() > 0]

train_texts = train_df['document'].tolist()
train_labels = train_df['label'].tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    train_texts,
    train_labels,
    test_size=0.1,      # train : val = 9:1
    random_state=1004,
    stratify=train_labels
)

tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=3,               # 전체 문서에서 등장 문서 수가 3 미만인 단어는 제거
        max_df=0.95,            # 너무 보편적인 단어(상위 5%)는 제거(정보량 낮음)
        ngram_range=(1,2),      # uni + bi-gram(문맥 반영)
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(
        max_iter=200,           # 최적화 반복 횟수
        C=4.0,                  # C=1이 디폴트, 규제 강도(작을 수록 규제 강함). 4.0은 비교적 규제가 약한 편.
        # n_jobs=None if hasattr(LogisticRegression(), "n_jobs") else None
    ))
])

tfidf_lr.fit(X_train, y_train)

# 검증셋 평가
valid_pred = tfidf_lr.predict(X_valid)
print("Validation accuracy:", accuracy_score(y_valid, valid_pred))
print(classification_report(y_valid, valid_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_valid, valid_pred))

test_pred = tfidf_lr.predict(test_df['document'])
print("Test accuracy:", accuracy_score(test_df['label'], test_pred))
print(classification_report(test_df['label'], test_pred, digits=4))

def predict_sentiment(text: str):
    cleaned = clean_korean(text)
    proba = tfidf_lr.predict_proba([cleaned])[0]
    label = int(proba[1] >= 0.5)
    return {"text": text, "pred": label, "neg_prob": float(proba[0]), "pos_prob": float(proba[1])}

examples = [
    "진짜 재밌고 감동적이었습니다. 배우들 연기 최고!!!",
    "시간 아깝고 졸리고, 핵노잼. 절대 안 봐. 뻐큐",
    "그럭저럭 볼만했지만 그닥 추천은 안함",
    "100억 갖다 버리는게 더 재밌겠다."
]
for ex in examples:
    print(predict_sentiment(ex))


Validation accuracy: 0.8116561956957626
              precision    recall  f1-score   support

           0     0.7882    0.8534    0.8195      7498
           1     0.8394    0.7697    0.8030      7464

    accuracy                         0.8117     14962
   macro avg     0.8138    0.8116    0.8113     14962
weighted avg     0.8138    0.8117    0.8113     14962

Confusion Matrix:
 [[6399 1099]
 [1719 5745]]
Test accuracy: 0.8138578507091416
              precision    recall  f1-score   support

           0     0.7904    0.8507    0.8194     24748
           1     0.8408    0.7776    0.8079     25101

    accuracy                         0.8139     49849
   macro avg     0.8156    0.8141    0.8137     49849
weighted avg     0.8158    0.8139    0.8136     49849

{'text': '진짜 재밌고 감동적이었습니다. 배우들 연기 최고!!!', 'pred': 1, 'neg_prob': 0.00811971166932246, 'pos_prob': 0.9918802883306775}
{'text': '시간 아깝고 졸리고, 핵노잼. 절대 안 봐. 뻐큐', 'pred': 0, 'neg_prob': 0.9927821052548912, 'pos_prob': 0.00721789474

In [ ]:
print("GPU 사용 가능 여부:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


GPU 사용 가능 여부: True
cuda


In [ ]:
frac = 0.4
train_df, _ = train_test_split(
    train_df,
    train_size=frac,
    stratify=train_df["label"],
    random_state=1004
)

MODEL_NAME = "beomi/KcELECTRA-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
if torch.cuda.is_available():
    model.to("cuda")
def tokenize_fn(batch):
    return tokenizer(batch["document"], padding="max_length", truncation=True, max_length=96)

dataset = DatasetDict(
    train=Dataset.from_pandas(train_df[["document","label"]]),
    test =Dataset.from_pandas(test_df[["document","label"]])
)
tokenized = dataset.map(tokenize_fn, batched=True, num_proc=2, remove_columns=["document"])

accuracy = evaluate.load("accuracy"); f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"acc": accuracy.compute(predictions=preds, references=labels)["accuracy"],
            "f1":  f1.compute(predictions=preds, references=labels, average="macro")["f1"]}

# --- 빠른 학습 설정 ---
training_args = TrainingArguments(
    output_dir="./out_fast",
    num_train_epochs=2,
    per_device_train_batch_size=32,    # T4면 32~64 사이에서 맞추기
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    eval_strategy="no",
    save_strategy="no",
    max_steps=800,
    fp16=True,  
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],    # 최종만 보고 싶으면 evaluate()만 호출
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
metrics = trainer.evaluate(tokenized["test"])

id2label = {0: "NEG", 1: "POS"}

def infer_transformer(texts):
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    if torch.cuda.is_available():
        model.to("cuda")
        enc = {k: v.to("cuda") for k, v in enc.items()}
    with torch.no_grad():
        model_inputs = {k: v for k, v in enc.items() if k != 'token_type_ids' or 'token_type_ids' in model.forward.__code__.co_varnames}
        out = model(**model_inputs).logits
        prob = out.softmax(dim=-1).cpu().numpy()
    preds = prob.argmax(axis=1)
    return [{"text": t, "pred": int(p), "label": id2label[int(p)], "neg_prob": float(pr[0]), "pos_prob": float(pr[1])}
            for t, p, pr in zip(texts, preds, prob)]

infer_transformer([
"진짜 재밌고 감동적이었습니다. 배우들 연기 최고!!!",
    "시간 아깝고 졸리고, 핵노잼. 절대 안 봐. 뻐큐",
    "그럭저럭 볼만했지만 그닥 추천은 안함",
    "100억 갖다 버리는게 더 재밌겠다."
])

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map (num_proc=2):   0%|          | 0/59847 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/49849 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
200,0.451300
400,0.344700
600,0.305700
800,0.274300


[{'text': '진짜 재밌고 감동적이었습니다. 배우들 연기 최고!!!',
  'pred': 1,
  'label': 'POS',
  'neg_prob': 0.011758035980165005,
  'pos_prob': 0.988241970539093},
 {'text': '시간 아깝고 졸리고, 핵노잼. 절대 안 봐. 뻐큐',
  'pred': 0,
  'label': 'NEG',
  'neg_prob': 0.9826347231864929,
  'pos_prob': 0.01736525259912014},
 {'text': '그럭저럭 볼만했지만 그닥 추천은 안함',
  'pred': 0,
  'label': 'NEG',
  'neg_prob': 0.9707630276679993,
  'pos_prob': 0.029237020760774612},
 {'text': '100억 갖다 버리는게 더 재밌겠다.',
  'pred': 0,
  'label': 'NEG',
  'neg_prob': 0.9798762798309326,
  'pos_prob': 0.02012374810874462}]

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. 경로 설정 (구글 드라이브 내부에 폴더 생성)
drive_artifacts_dir = "/content/drive/MyDrive/artifacts"
drive_model_dir = "/content/drive/MyDrive/saved_model_nsmc"
os.makedirs(drive_artifacts_dir, exist_ok=True)
os.makedirs(drive_model_dir, exist_ok=True)
# 2. 전통 ML 모델을 구글 드라이브의 artifacts 폴더에 저장
joblib.dump(tfidf_lr, f"{drive_artifacts_dir}/tfidf_lr_nsms.joblib")
print("TF-IDF + Logistic Regression 모델 저장 완료 (Google Drive)")
# 3. 트랜스포머 모델 및 토크나이저를 구글 드라이브의 saved_model_nsmc 폴더에 저장
trainer.save_model(drive_model_dir)
tokenizer.save_pretrained(drive_model_dir)
print(f"트랜스포머 모델과 토크나이저가 드라이브에 저장되었습니다: {drive_model_dir}")

TF-IDF + Logistic Regression 모델 저장 완료
트랜스포머 모델과 토크나이저가 저장되었습니다: ./saved_model_nsmc


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def clean_document(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compression_ratio(source: str, summary: str) -> float:
    if len(source) == 0:
        return 0.0
    return round(len(summary) / len(source), 3)

def show_summary(source: str, summary: str) -> None:
    print("[원문]")
    print(source)
    print("\n[요약문]")
    print(summary)
    print("\n원문 글자 수:", len(source))
    print("요약문 글자 수:", len(summary))
    print("압축률:", compression_ratio(source, summary))

english_text = """
Text mining, also referred to as text data mining, is the process of deriving high-quality information from text.
It involves the discovery by computer of new, previously unknown information by automatically extracting information from different written resources.
Written resources may include websites, books, emails, reviews, and articles.
High-quality information is typically obtained by devising patterns and trends by means such as statistical pattern learning.
Text mining usually involves structuring the input text, deriving patterns within the structured data, and finally evaluating and interpreting the output.
"""

korean_text = """
디아블로는 액션 롤플레잉 핵 앤드 슬래시 비디오 게임이다.
플레이어는 주변 환경을 마우스로 사용해 영웅을 움직이게 한다.
주문을 외는 등의 다른 활동은 키보드 입력으로 이루어진다.
플레이어는 이 게임에서 장비를 획득하고, 주문을 배우고, 적을 쓰러뜨리며, NPC와 대화를 나눌 수 있다.
지하 미궁은 주어진 형식이 있고 부분적으로 반복되는 형태가 존재하나 전체적으로 보면 무작위로 생성된다.
예를 들어 지하 묘지의 경우에는 긴 복도와 닫힌 문들이 존재하고, 동굴은 좀 더 선형 형태를 띠고 있다.
플레이어에게는 몇몇 단계에서 무작위의 퀘스트를 받는다.
이 퀘스트는 선택적인 사항이나 플레이어의 영웅들을 성장시키거나 줄거리를 이해하는데 도움을 준다.
그러나 맨 뒤에 두 퀘스트는 게임을 끝내기 위해 완료시켜야 한다.
"""

english_text = clean_document(english_text)
korean_text = clean_document(korean_text)

print("영문 예제 길이:", len(english_text))
print("한글 예제 길이:", len(korean_text))

def summarize_with_seq2seq(
    text: str,
    tokenizer,
    model,
    prefix: str = "summarize: ",
    max_input_length: int = 512,
    min_length: int = 30,
    max_length: int = 100,
    num_beams: int = 4,
    no_repeat_ngram_size: int = 3,
    length_penalty: float = 1.0,  # 1. 1.0이 기본값인 파라미터 추가
) -> str:
    source = clean_document(text)
    input_text = prefix + source if prefix else source
    encoded = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    ).to(model.device)

    # BART models do not accept token_type_ids in generate().
    encoded.pop("token_type_ids", None)

    generation_kwargs = {
        "num_beams": num_beams,
        "no_repeat_ngram_size": no_repeat_ngram_size,
        "min_length": min_length,
        "max_length": max_length,
        "length_penalty": length_penalty,  # 2. generation_kwargs에 인자 추가
    }
    if num_beams > 1:
        generation_kwargs["early_stopping"] = True

    with torch.inference_mode():
        summary_ids = model.generate(**encoded, **generation_kwargs)

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

baseline_model_name = 'sshleifer/distilbart-cnn-12-6'

# 일부 transformers 설치 환경에서는 pipeline registry에 "summarization" 태스크가 없을 수 있습니다.
# 그 경우 같은 모델을 AutoModelForSeq2SeqLM.generate()로 직접 호출합니다.
try:
    summarizer = pipeline(
        task="summarization",
        model=baseline_model_name,
        device=0 if torch.cuda.is_available() else -1,
    )
    pipeline_result = summarizer(
        english_text,
        min_length=25,
        max_length=90,
        do_sample=False,
    )
    pipeline_summary = pipeline_result[0]["summary_text"]
    print("pipeline 방식으로 요약했습니다.")

except KeyError as exc:
    print("현재 transformers 환경에서 summarization pipeline을 사용할 수 없어 직접 generate 방식으로 대체합니다.")
    print("원인:", exc)

    baseline_tokenizer = AutoTokenizer.from_pretrained(baseline_model_name)
    baseline_model = AutoModelForSeq2SeqLM.from_pretrained(baseline_model_name).to(device)
    baseline_model.eval()

    encoded = baseline_tokenizer(
        english_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    with torch.inference_mode():
        summary_ids = baseline_model.generate(
            **encoded,
            min_length=25,
            max_length=90,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    pipeline_summary = baseline_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

show_summary(english_text, pipeline_summary)

영문 예제 길이: 623
한글 예제 길이: 399
현재 transformers 환경에서 summarization pipeline을 사용할 수 없어 직접 generate 방식으로 대체합니다.
원인: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"


[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

[원문]
Text mining, also referred to as text data mining, is the process of deriving high-quality information from text. It involves the discovery by computer of new, previously unknown information by automatically extracting information from different written resources. Written resources may include websites, books, emails, reviews, and articles. High-quality information is typically obtained by devising patterns and trends by means such as statistical pattern learning. Text mining usually involves structuring the input text, deriving patterns within the structured data, and finally evaluating and interpreting the output.

[요약문]
 Text mining involves deriving high-quality information from text . Written resources may include websites, books, emails, reviews, and articles . Text mining usually involves structuring the input text, deriving patterns within the structured data, and finally evaluating and interpreting the output .

원문 글자 수: 623
요약문 글자 수: 302
압축률: 0.485


In [3]:
t5_model_name = 't5-small'

t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name, model_max_length=512)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name).to(device)
t5_model.eval()

print("tokenizer type:", type(t5_tokenizer))
print("model type:", type(t5_model))

t5_summary = summarize_with_seq2seq(
    english_text,
    tokenizer=t5_tokenizer,
    model=t5_model,
    prefix="summarize: ",
    min_length=30,
    max_length=100,
)

show_summary(english_text, t5_summary)

option_rows = []
for beams in [1, 2, 4]:
    summary = summarize_with_seq2seq(
        english_text,
        tokenizer=t5_tokenizer,
        model=t5_model,
        prefix="summarize: ",
        min_length=25,
        max_length=80,
        num_beams=beams,
    )
    option_rows.append({
        "num_beams": beams,
        "summary": summary,
        "summary_length": len(summary),
        "compression_ratio": compression_ratio(english_text, summary),
    })

pd.DataFrame(option_rows)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

tokenizer type: <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>
model type: <class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>
[원문]
Text mining, also referred to as text data mining, is the process of deriving high-quality information from text. It involves the discovery by computer of new, previously unknown information by automatically extracting information from different written resources. Written resources may include websites, books, emails, reviews, and articles. High-quality information is typically obtained by devising patterns and trends by means such as statistical pattern learning. Text mining usually involves structuring the input text, deriving patterns within the structured data, and finally evaluating and interpreting the output.

[요약문]
text mining is the process of deriving high-quality information from text. it involves the discovery by computer of new, previously unknown information. written resources may include websites, books, ema

,num_beams,summary,summary_length,compression_ratio
0,1,text mining is the process of deriving high-qu...,230,0.369
1,2,text mining is the process of deriving high-qu...,230,0.369
2,4,text mining is the process of deriving high-qu...,230,0.369


In [4]:
kobart_model_name = 'gogamza/kobart-summarization'

kobart_tokenizer = PreTrainedTokenizerFast.from_pretrained(kobart_model_name)
kobart_model = BartForConditionalGeneration.from_pretrained(kobart_model_name).to(device)
kobart_model.eval()

kobart_summary = summarize_with_seq2seq(
    korean_text,
    tokenizer=kobart_tokenizer,
    model=kobart_model,
    prefix="",
    max_input_length=1024,
    min_length=50,
    max_length=120,
    num_beams=4,
    no_repeat_ngram_size=3,
    length_penalty=1.5,     # 길게 생성할수록 가점 부여
)

show_summary(korean_text, kobart_summary)

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

[원문]
디아블로는 액션 롤플레잉 핵 앤드 슬래시 비디오 게임이다. 플레이어는 주변 환경을 마우스로 사용해 영웅을 움직이게 한다. 주문을 외는 등의 다른 활동은 키보드 입력으로 이루어진다. 플레이어는 이 게임에서 장비를 획득하고, 주문을 배우고, 적을 쓰러뜨리며, NPC와 대화를 나눌 수 있다. 지하 미궁은 주어진 형식이 있고 부분적으로 반복되는 형태가 존재하나 전체적으로 보면 무작위로 생성된다. 예를 들어 지하 묘지의 경우에는 긴 복도와 닫힌 문들이 존재하고, 동굴은 좀 더 선형 형태를 띠고 있다. 플레이어에게는 몇몇 단계에서 무작위의 퀘스트를 받는다. 이 퀘스트는 선택적인 사항이나 플레이어의 영웅들을 성장시키거나 줄거리를 이해하는데 도움을 준다. 그러나 맨 뒤에 두 퀘스트는 게임을 끝내기 위해 완료시켜야 한다.

[요약문]
액션 롤플플레잉 핵 앤드 슬래시 비디오 게임인 디아블로는 액션 롤 플레잉핵 앤드슬래시비디오 게임인 핵 앤 드 슬래시가 비디오 게임으로 플레이어는 주변 환경을 마우스로 사용해 영웅을 움직이게 하고, 주문을 외는 등의 다른 활동은 키보드 입력으로 이루어진다.

원문 글자 수: 399
요약문 글자 수: 142
압축률: 0.356


In [5]:
mt5_model_name = 'csebuetnlp/mT5_multilingual_XLSum'

# mT5는 SentencePiece 기반 토크나이저
mt5_tokenizer = AutoTokenizer.from_pretrained(mt5_model_name, use_fast=False)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(mt5_model_name).to(device)
mt5_model.eval()

mt5_summary = summarize_with_seq2seq(
    korean_text,
    tokenizer=mt5_tokenizer,
    model=mt5_model,
    prefix="",
    max_input_length=1024,
    min_length=15,
    max_length=120,
    num_beams=4,
    no_repeat_ngram_size=2,
)

show_summary(korean_text, mt5_summary)

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[원문]
디아블로는 액션 롤플레잉 핵 앤드 슬래시 비디오 게임이다. 플레이어는 주변 환경을 마우스로 사용해 영웅을 움직이게 한다. 주문을 외는 등의 다른 활동은 키보드 입력으로 이루어진다. 플레이어는 이 게임에서 장비를 획득하고, 주문을 배우고, 적을 쓰러뜨리며, NPC와 대화를 나눌 수 있다. 지하 미궁은 주어진 형식이 있고 부분적으로 반복되는 형태가 존재하나 전체적으로 보면 무작위로 생성된다. 예를 들어 지하 묘지의 경우에는 긴 복도와 닫힌 문들이 존재하고, 동굴은 좀 더 선형 형태를 띠고 있다. 플레이어에게는 몇몇 단계에서 무작위의 퀘스트를 받는다. 이 퀘스트는 선택적인 사항이나 플레이어의 영웅들을 성장시키거나 줄거리를 이해하는데 도움을 준다. 그러나 맨 뒤에 두 퀘스트는 게임을 끝내기 위해 완료시켜야 한다.

[요약문]
디아블로 게임을 소개한다. 지하 미궁과 동굴 등 다양한 형태의 게임이다.

원문 글자 수: 399
요약문 글자 수: 40
압축률: 0.1


In [6]:
comparison_df = pd.DataFrame([
    {
        "model": "pipeline-distilbart",
        "language": "English",
        "summary": pipeline_summary,
        "source_length": len(english_text),
        "summary_length": len(pipeline_summary),
        "compression_ratio": compression_ratio(english_text, pipeline_summary),
    },
    {
        "model": "t5-small",
        "language": "English",
        "summary": t5_summary,
        "source_length": len(english_text),
        "summary_length": len(t5_summary),
        "compression_ratio": compression_ratio(english_text, t5_summary),
    },
    {
        "model": "KoBART",
        "language": "Korean",
        "summary": kobart_summary,
        "source_length": len(korean_text),
        "summary_length": len(kobart_summary),
        "compression_ratio": compression_ratio(korean_text, kobart_summary),
    },
    {
        "model": "mT5-XLSum",
        "language": "Korean",
        "summary": mt5_summary,
        "source_length": len(korean_text),
        "summary_length": len(mt5_summary),
        "compression_ratio": compression_ratio(korean_text, mt5_summary),
    },
])

comparison_df

,model,language,summary,source_length,summary_length,compression_ratio
0,pipeline-distilbart,English,Text mining involves deriving high-quality in...,623,302,0.485
1,t5-small,English,text mining is the process of deriving high-qu...,623,230,0.369
2,KoBART,Korean,액션 롤플플레잉 핵 앤드 슬래시 비디오 게임인 디아블로는 액션 롤 플레잉핵 앤드슬래...,399,142,0.356
3,mT5-XLSum,Korean,디아블로 게임을 소개한다. 지하 미궁과 동굴 등 다양한 형태의 게임이다.,399,40,0.100


In [7]:
import pandas as pd
from kiwipiepy import Kiwi
from rouge_score import rouge_scorer

# 1. 정답 요약문 정의
korean_reference_summary = "디아블로는 장비 획득, 전투, 퀘스트 수행을 중심으로 진행되는 액션 롤플레잉 게임이며, 지하 미궁과 퀘스트는 게임 진행과 성장에 중요한 역할을 한다."

# 2. 한국어 형태소 분석기 초기화 및 분리 함수
kiwi = Kiwi()

def tokenize_korean_morphs(text: str) -> str:
    tokens = kiwi.tokenize(text)
    return " ".join([t.form for t in tokens])

# 3. 한글 유니코드 드롭 버그를 우회하는 ROUGE 산출 함수
def compute_korean_rouge_scores(ref_text: str, pred_text: str) -> dict:
    # 형태소 분리 수행
    tokens_ref = tokenize_korean_morphs(ref_text).split()
    tokens_pred = tokenize_korean_morphs(pred_text).split()
    
    # 한글 단어별 가상 영어 ID 사영 (Dictionary) 생성하여 한글 검열 버그 우회
    word_to_id = {}
    id_counter = 0
    
    def translate_to_eng_ids(tokens):
        nonlocal id_counter
        translated = []
        for t in tokens:
            if t not in word_to_id:
                word_to_id[t] = f"w{id_counter}"
                id_counter += 1
            translated.append(word_to_id[t])
        return " ".join(translated)
    
    eng_ref = translate_to_eng_ids(tokens_ref)
    eng_pred = translate_to_eng_ids(tokens_pred)
    
    # 구글 rouge_score 계산 실행
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    raw_scores = scorer.score(eng_ref, eng_pred)
    
    # F1-Score만 추출하여 반환
    return {
        "rouge1": round(raw_scores["rouge1"].fmeasure, 4),
        "rouge2": round(raw_scores["rouge2"].fmeasure, 4),
        "rougeL": round(raw_scores["rougeL"].fmeasure, 4)
    }

# 4. 모델별 평가 데이터 수집 및 데이터프레임 빌드
rouge_results = []
for model_name, summary in [
    ("KoBART", kobart_summary),
    ("mT5-XLSum", mt5_summary),
]:
    scores = compute_korean_rouge_scores(korean_reference_summary, summary)
    rouge_results.append({
        "model": model_name,
        **scores
    })

# 최종 결과 테이블 출력
pd.DataFrame(rouge_results)


,model,rouge1,rouge2,rougeL
0,KoBART,0.3505,0.0842,0.2268
1,mT5-XLSum,0.4333,0.2069,0.2667


In [8]:
from datasets import Dataset, load_dataset

def build_mini_summary_dataset() -> Dataset:
    """Hugging Face BillSum 로딩이 실패할 때 쓰는 수업용 미니 데이터셋."""
    rows = [
        {
            "title": "Clean Energy Investment Act",
            "text": "The bill establishes a grant program to support clean energy projects in public schools. It prioritizes schools in low-income communities and requires annual reporting on energy savings, emissions reductions, and student health benefits.",
            "summary": "Creates grants for clean energy projects in public schools, prioritizing low-income communities and requiring annual reports.",
        },
        {
            "title": "Small Business Digital Training Act",
            "text": "This act directs the commerce department to provide digital skills training for small businesses. The program includes cybersecurity basics, online marketing, electronic payments, and data privacy education.",
            "summary": "Directs the commerce department to provide digital skills and cybersecurity training for small businesses.",
        },
        {
            "title": "Rural Hospital Support Act",
            "text": "The legislation creates temporary funding for rural hospitals facing staffing shortages. Hospitals receiving funds must report how the money is used to retain nurses, expand telehealth access, and maintain emergency care.",
            "summary": "Provides temporary funding for rural hospitals to retain staff, expand telehealth, and maintain emergency care.",
        },
        {
            "title": "Student Nutrition Improvement Act",
            "text": "The bill expands school meal reimbursement rates and supports local food purchasing. It also requires nutrition education materials to be made available to families in multiple languages.",
            "summary": "Expands school meal support, encourages local food purchasing, and requires multilingual nutrition education materials.",
        },
        {
            "title": "Public Transit Safety Act",
            "text": "This measure requires transit agencies to update safety plans, improve operator training, and report major incidents to a national database. Grants are authorized for camera systems and station lighting.",
            "summary": "Requires updated transit safety plans, improved training, incident reporting, and grants for cameras and lighting.",
        },
        {
            "title": "Water Infrastructure Resilience Act",
            "text": "The act authorizes funding for local water systems to replace aging pipes and prepare for droughts and floods. Priority is given to communities with repeated service disruptions.",
            "summary": "Funds water system upgrades for aging pipes and climate resilience, prioritizing communities with repeated disruptions.",
        },
        {
            "title": "Veterans Employment Services Act",
            "text": "The bill expands job placement services for veterans transitioning to civilian work. It creates partnerships with community colleges and employers in health care, logistics, and advanced manufacturing.",
            "summary": "Expands veteran job placement services through partnerships with colleges and employers in key industries.",
        },
        {
            "title": "Child Care Workforce Act",
            "text": "This act provides grants to states to improve wages and training for child care workers. States must submit plans describing how funds will increase workforce retention and access for families.",
            "summary": "Provides state grants to improve child care worker wages, training, retention, and family access.",
        },
        {
            "title": "Wildfire Preparedness Act",
            "text": "The legislation supports wildfire prevention by funding forest management, emergency communication systems, and evacuation planning. Local governments must coordinate with tribal and regional agencies.",
            "summary": "Funds wildfire prevention, emergency communication, and evacuation planning with regional coordination requirements.",
        },
        {
            "title": "Affordable Housing Preservation Act",
            "text": "The bill creates loans and tax incentives to preserve affordable rental housing. Property owners receiving support must maintain affordability requirements for a minimum period.",
            "summary": "Creates loans and tax incentives to preserve affordable rental housing with long-term affordability requirements.",
        },
    ]
    return Dataset.from_list(rows)


try:
    billsum_raw = load_dataset("billsum", split="ca_test[:200]")
    dataset_source = "Hugging Face billsum"
except Exception as exc:
    print("BillSum 로딩 실패. Colab의 datasets/huggingface_hub 호환성 문제일 수 있습니다.")
    print("미니 요약 데이터셋으로 실습을 계속합니다.")
    print("오류:", type(exc).__name__, exc)
    billsum_raw = build_mini_summary_dataset()
    dataset_source = "local mini summary dataset"

billsum = billsum_raw.train_test_split(test_size=0.2, seed=42)

example = billsum["train"][0]
print("데이터 출처:", dataset_source)
print("Train/Test 크기:", len(billsum["train"]), len(billsum["test"]))
print("\n요약 데이터 예")
print("Title:", example["title"][:120])
print("Text:", example["text"][:300])
print("Summary:", example["summary"][:300])

train_tokenizer = AutoTokenizer.from_pretrained("t5-small", model_max_length=512)
train_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(device)


def preprocess_billsum(batch):
    inputs = ["summarize: " + clean_document(doc) for doc in batch["text"]]
    model_inputs = train_tokenizer(
        inputs,
        max_length=512,
        truncation=True,
    )
    labels = train_tokenizer(
        text_target=batch["summary"],
        max_length=128,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_billsum = billsum.map(
    preprocess_billsum,
    batched=True,
    remove_columns=billsum["train"].column_names,
)

tokenized_billsum

BillSum 로딩 실패. Colab의 datasets/huggingface_hub 호환성 문제일 수 있습니다.
미니 요약 데이터셋으로 실습을 계속합니다.
오류: HfUriError Invalid HF URI 'hf://datasets/billsum@3d8510441c06a3d9dfb32eb0d7f80151730bcc4f/.huggingface.yaml'. Repository id must be 'namespace/name', got 'billsum'.
데이터 출처: local mini summary dataset
Train/Test 크기: 8 2

요약 데이터 예
Title: Clean Energy Investment Act
Text: The bill establishes a grant program to support clean energy projects in public schools. It prioritizes schools in low-income communities and requires annual reporting on energy savings, emissions reductions, and student health benefits.
Summary: Creates grants for clean energy projects in public schools, prioritizing low-income communities and requiring annual reports.


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2
    })
})

In [10]:
from inspect import signature

from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

data_collator = DataCollatorForSeq2Seq(tokenizer=train_tokenizer, model=train_model)

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = train_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, train_tokenizer.pad_token_id)
    decoded_labels = train_tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {key: round(value, 4) for key, value in result.items()}


args_kwargs = {
    "output_dir": "./summary_tutor",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "weight_decay": 0.01,
    "save_total_limit": 2,
    "max_steps": 20,
    "predict_with_generate": True,
    "logging_steps": 5,
    "save_steps": 20,
    "report_to": "none",
}

if "eval_strategy" in signature(Seq2SeqTrainingArguments).parameters:
    args_kwargs["eval_strategy"] = "steps"
else:
    args_kwargs["evaluation_strategy"] = "steps"
args_kwargs["eval_steps"] = 20

training_args = Seq2SeqTrainingArguments(**args_kwargs)

trainer = Seq2SeqTrainer(
    model=train_model,
    args=training_args,
    train_dataset=tokenized_billsum["train"],
    eval_dataset=tokenized_billsum["test"],
    processing_class=train_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
20,1.969238,2.901109,0.404400,0.180400,0.336400,0.336400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Rouge1,Rouge2,Rougel,Rougelsum
1.969238,2.901109,20,0.404400,0.180400,0.336400,0.336400


In [ ]:
save_dir = "./saved_summary_t5"
trainer.save_model(save_dir)
train_tokenizer.save_pretrained(save_dir)
print(f"요약 모델과 토크나이저가 저장되었습니다: {save_dir}")

In [ ]:
loaded_tokenizer = AutoTokenizer.from_pretrained(save_dir)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(save_dir).to(device)
loaded_model.eval()

bill_text = clean_document(billsum["test"][0]["text"])
bill_gold = billsum["test"][0]["summary"]

loaded_summary = summarize_with_seq2seq(
    bill_text,
    tokenizer=loaded_tokenizer,
    model=loaded_model,
    prefix="summarize: ",
    max_input_length=512,
    min_length=30,
    max_length=120,
)

print("[Gold Summary]")
print(bill_gold[:700])
print("\n[Generated Summary]")
print(loaded_summary)

OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: '<transformers.trainer_seq2seq.Seq2SeqTrainer object at 0x707dc60ec7d0>'.

In [13]:
from config import CONTENT_DIR
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# 문자열로 저장된 리스트 형식을 실제 리스트로 변환하기 위해 사용
from ast import literal_eval

movie_df = pd.read_csv(CONTENT_DIR / 'tmdb_5000_movies.csv')
# 필요한 컬럼만 추출
movie_df = movie_df[['id', 'title', 'genres', 'vote_average', "vote_count", 'popularity', 'keywords', 'overview']]
# 최대 400자까지 보겠다.
pd.set_option('max_colwidth', 400)
movie_df[['genres', 'keywords']][:1]
movie_df['genres'] = movie_df['genres'].apply(literal_eval)
movie_df['keywords'] = movie_df['keywords'].apply(literal_eval)
movie_df[['genres', 'keywords']].iloc[0]
genres = []         # 모든 영화의 장르 이름을 담을 리스트

for tmp_genres in movie_df['genres']:
    tmpList = []        # 각 영화의 장르 이름을 임시로 저장
    for i in tmp_genres:
        tmpList.append(i['name'])    # name 키에 해당하는 값을 추출
    genres.append(tmpList)

movie_df['genres'] = genres
# keywords 컬럼도 각 영화의 키워드 이름만 추출
movie_df['keywords'] = movie_df['keywords'].apply(lambda x: [i['name'] for i in x])
# keywords, genres 컬럼의 리스트 데이터를 공백을 기준으로 하나의 문자열로 변환
movie_df['genres'] = movie_df['genres'].apply(lambda x: ' '.join(x))
movie_df['keywords'] = movie_df['keywords'].apply(lambda x: ' '.join(x))

# genres 컬럼을 수치 벡터로 변환. 문장에서 등장하는 단어의 빈도를 기반으로 행렬을 생성
count_vect = CountVectorizer(min_df=1,          # 최소 한번이라도 등장한 단어만 벡터 포함
                             ngram_range=(1,2)) # 단어1, 단어 2개까지 고려

genres_mat = count_vect.fit_transform(movie_df['genres'])

# 장르 벡터 행렬 사이의 코사인 유사도 계산
# 모든 영화 장르를 벡터화 한 후 자기 자신과 비교해서 영화 간 유사도 행렬 생성
genres_sim = cosine_similarity(genres_mat, genres_mat)

genres_sim.argsort()[:,::-1]   # 내림차순 정렬, 유사도가 높은 순으로 정렬
genres_sim_sorted_ind = genres_sim.argsort()[:, ::-1]

# 유사 영화 찾기 함수(입력된 영화와 유사도가 높은 영화를 찾는다.)
def find_sim_movie(df, sorted_ind, title_name, top_n=10):

    # 입력된 영화 제목으로 해당 영화의 데이터와 인덱스를 찾음
    title_movie = df[df['title'] == title_name]

    # 해당 영화의 인덱스 값을 가져옴
    title_index = title_movie.index.values

    # 해당 영화와 유사한 영화들의 인덱스를 가져옴
    similar_indexes = sorted_ind[title_index, :top_n]
    print(similar_indexes)      # 확인
    similar_indexes = similar_indexes.reshape(-1)

    # 해당 인덱스에 해당하는 영화 리턴
    return df.iloc[similar_indexes]

similar_movies = find_sim_movie(movie_df, genres_sim_sorted_ind, 'The Godfather', 10)       # The Godfather와 가장 유사한 영화 10편 찾아줘.
display(similar_movies[['title', 'vote_average']])
percetile = 0.6         # 상위 60% 기준
m = movie_df['vote_count'].quantile(percetile)      # 370 : 평가 수가 370개 이상인 영화만 신뢰할 수 있는 영화로 간주
c = movie_df['vote_average'].quantile(percetile)    # 6.5 : 평균 평점이 6.5 이상인 영화들을 추천 대상으로 고려

def weight_vote_average(record):
    v = record['vote_count']        # 특정 영화의 평가수
    r = record['vote_average']      # 특정 영화의 평균 평점

    # IMDB에서 평가 횟수에 대한 가중치가 부여된 평점 방식
    score = ((v/(v+m))*r) + ((m/(m+v))*c)

    return score

# 가중 평점을 컬럼에 저장
movie_df['weight_vote_average'] = movie_df.apply(weight_vote_average, axis=1)

# 가중 평점 기반 영화 추천
def find_sim_movie(df, sorted_ind, title_name, top_n=10):
    title_movie = df[df['title'] == title_name]
    title_index = title_movie.index.values

    # 유사도가 높은 영화 인덱스 추출(top_n의 두배에 해당되는)
    similar_indexes = sorted_ind[title_index, :(top_n*2)]
    similar_indexes = similar_indexes.reshape(-1)

    # 기존 영화 index 제외
    similar_indexes = similar_indexes[similar_indexes != title_index]

    return df.iloc[similar_indexes].sort_values('weight_vote_average', ascending = False)[:top_n]

find_sim_movies = find_sim_movie(movie_df, genres_sim_sorted_ind, 'Avatar', 10)
find_sim_movies[['title', 'vote_average', 'weight_vote_average']]

[[ 281 2839 4217  883  892 3378 3112 3636 1370 1847]]


,title,vote_average
281,American Gangster,7.4
2839,Rounders,6.9
4217,Kids,6.8
883,Catch Me If You Can,7.7
892,Casino,7.8
3378,Auto Focus,6.1
3112,Blood Done Sign My Name,6.0
3636,Light Sleeper,5.7
1370,21,6.5
1847,GoodFellas,8.2


,title,vote_average,weight_vote_average
46,X-Men: Days of Future Past,7.5,7.442176
813,Superman,6.9,6.793636
3208,Star Wars: Clone Wars: Volume 1,8.0,6.601964
870,Superman II,6.5,6.500000
14,Man of Steel,6.5,6.500000
420,Hellboy II: The Golden Army,6.5,6.500000
3494,Beastmaster 2: Through the Portal of Time,4.6,6.416581
1932,Sheena,5.0,6.415859
1191,Small Soldiers,6.2,6.326033
232,The Wolverine,6.3,6.316739
